In [22]:
import pandas as pd
import numpy as np

from tgb.nodeproppred.dataset import NodePropPredDataset

dataset = NodePropPredDataset(name='tgbn-13f', root='datasets')
node_ids = dataset.node_ids
df_cik = pd.read_csv('cik_features.csv')
df_cusip = pd.read_csv('cusip_features.csv')
df_cik_columns = list(df_cik.columns)
num_cik_features = len(df_cik_columns) - 2  # Exclude 'CIK' and 'source_file'
cik_values = df_cik['CIK'].values
#df_cik_columns.remove('CIK')
#df_cik_columns.remove('source_file')
df_cusip_columns = list(df_cusip.columns)
num_cusip_features = len(df_cusip_columns) - 2  # Exclude 'CUSIP' and 'source_file'
cusip_values = df_cusip['CUSIP'].values
#df_cusip_columns.remove('CUSIP')
#df_cusip_columns.remove('source_file')
col_names_combined = df_cik_columns + df_cusip_columns
df_node = pd.DataFrame(columns=col_names_combined)
#print(df_node.head())
node_features = np.zeros((48, 602, 23))
#df_fixed = df_cusip[df_cusip['source_file'] == '20251'].copy()
#fixed_values = df_cusip[df_cusip['source_file'] == '20132']['sector'].values
#node_features[:, :, 22] = fixed_values
source = dataset.label_ts
#print(source)
source_id = {s: i for i, s in zip(range(len(source)), source)}
for s in source:
    for node_name, node_id in node_ids.items():
        if node_name in cik_values:
            v = df_cik[(df_cik['CIK'] == node_name) &
                              (df_cik['source_file'] == s)].iloc[:, 2:].values
            if v.size != 0:
                node_features[source_id[s], node_id,
                          num_cik_features:] = v
            # CUSIP part remains zero
        elif node_name in cusip_values:
            v = df_cusip[(df_cusip['CUSIP'] == node_name) &
                              (df_cusip['source_file'] == s)].iloc[:, 2:].values
            if v.size != 0:
                node_features[source_id[s], node_id,
                          :num_cusip_features] = v
            # CIK part remains zero
        # else: node is not in either, remains all zero
        
np.save('node_feat.npy', node_features)

raw file found, skipping download
Dataset directory is  c:\Users\emadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tgb/datasets\tgbn_13f
loading processed file


In [ ]:
import pandas as pd
import numpy as np

df_cik = pd.read_csv('cik_features.csv')
df_cusip = pd.read_csv('cusip_features.csv')
df_cik_columns = list(df_cik.columns)
df_cusip_columns = list(df_cusip.columns)
num_nodes = len(node_ids)
cik_feat_dim = len(df_cik_columns) - 2
cusip_feat_dim = len(df_cusip_columns) - 2
feat_dim = cik_feat_dim + cusip_feat_dim

node_features = np.zeros((num_nodes, feat_dim))

for orig_id, idx in node_ids.items():
    if is_cik(orig_id):  # You need a function to check if orig_id is a CIK
        cik_vec = get_cik_features(orig_id)  # shape: (cik_feat_dim,)
        node_features[idx, :cik_feat_dim] = cik_vec
        # CUSIP part remains zero
    elif is_cusip(orig_id):
        cusip_vec = get_cusip_features(orig_id)  # shape: (cusip_feat_dim,)
        node_features[idx, cik_feat_dim:] = cusip_vec
        # CIK part remains zero

In [ ]:
import numpy as np

# Example: for a given timestamp
cik_features = ...  # dict {CIK: np.array([...])}
cusip_features = ...  # dict {CUSIP: np.array([...])}
num_cik_features = ...  # e.g., cik_features['...'].shape[0]
num_cusip_features = ...  # e.g., cusip_features['...'].shape[0]

node_features = np.zeros((num_nodes, num_cik_features + num_cusip_features))

for node_name, node_id in node_id_map.items():
    if node_name in cik_features:
        node_features[node_id, :num_cik_features] = cik_features[node_name]
        # CUSIP part remains zero
    elif node_name in cusip_features:
        node_features[node_id, num_cik_features:] = cusip_features[node_name]
        # CIK part remains zero
    # else: node is not in either, remains all zero